# Test Features/Functions
This is a temp notebook to test out functions and features required for the DECONVersation package. Using a step by step guide on how a user would use this package. <br>
<b> Note </b>: This might be converted to a tutorial upon package deployment

# Step 0: Prep Data

In [1]:
# For reading and processing pseudobulk
from pseudobulk import *

# For pre-processing h5ad for geneformer or cell2sentence
from preprocessing import *

In [2]:
# Path to pseudobulk
# Contains two samples that are essentially duplicates (pre-menopausal)
path_1 = "/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/single_cell_refs/CZI04N_post_menopausal_sequence_duplicates.h5ad"

# Contains single cell data from a single sample with batch effects (post-menopausal)
path_2 = "/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/single_cell_refs/CZI041N_pre_menopausal.h5ad"

In [3]:
# Cell lines 
path_3 = "/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/GSE220608/GSM6807287_cleanso.h5ad"

### For Geneformer

In [4]:
import scanpy as sc
# Read the h5ad file (just to explore columns and variables)
adata_1 = sc.read_h5ad(path_1)

# Prep data for geneformer (first data set)
adata_1_gf = adata_1.copy()

# Get ensembl ID
adata_1_gf.var.index = gene_id_name_map(gene_list=adata_1_gf.var.index,
                                      mode="to_symbol" )
# Remove unmapped genes
adata_1_gf = adata_1_gf[:, adata_1_gf.var.index.notnull()]

# Prep data
adata_1_gf = load_and_prep_data(adata= adata_1_gf, 
                              cell_type_col= "heca_lineage", 
                              mode="geneformer")

/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv/src/DECONVersation/preprocessing.py:49: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["cell_id"] = adata.obs.index


In [5]:
# Read the h5ad file (just to explore columns and variables)
adata_2 = sc.read_h5ad(path_2)

# Prep data for geneformer (first data set)
adata_2_gf = adata_2.copy()

# Get ensembl ID
adata_2_gf.var.index = gene_id_name_map(gene_list=adata_2_gf.var.index,
                                      mode="to_symbol" )
# Remove unmapped genes
adata_2_gf = adata_2_gf[:, adata_2_gf.var.index.notnull()]

# Prep data
adata_2_gf = load_and_prep_data(adata= adata_2_gf, 
                              cell_type_col= "heca_lineage", 
                              mode="geneformer")

/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv/src/DECONVersation/preprocessing.py:49: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["cell_id"] = adata.obs.index


In [8]:
# Read the h5ad file (just to explore columns and variables)
adata_3 = sc.read_h5ad(path_3)
adata_3 = adata_3[adata_3.obs["type"] != "hMSC"]
adata_3.var.index = adata_3.var["gene_name"]

#adata_3 = adata_3[adata_3.obs["orig.ident"].isin(["GSM6807289_M3", "GSM6807290_M4", "GSM6807291_M5"])]

# Prep data for geneformer (first data set)
adata_3_gf = adata_3.copy()

# Get ensembl ID
adata_3_gf.var.index = gene_id_name_map(gene_list=adata_3_gf.var.index,
                                    mode="to_symbol" )

# Remove unmapped genes
adata_3_gf = adata_3_gf[:, adata_3_gf.var.index.notnull()]

# Prep data
adata_3_gf = load_and_prep_data(adata= adata_3_gf, 
                              cell_type_col= "type", 
                              mode="geneformer")

/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv/src/DECONVersation/preprocessing.py:49: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["cell_id"] = adata.obs.index


In [9]:
print(adata_3)

View of AnnData object with n_obs × n_vars = 42369 × 36581
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'Percent_mito', 'qc', 'RNA_snn_res.0.1', 'seurat_clusters', 'type'
    var: 'gene_name'


### For Cell2Sentence

In [10]:
# Prep data for geneformer (first data set)
adata_1_c2s = adata_1.copy()

# Prep data for Cell2Sentence
adata_1_c2s = load_and_prep_data(adata = adata_1_c2s,
                                 cell_type_col = "heca_lineage", 
                                 mode = "c2s",
                                 organism = "Homo Sapiens")

In [11]:
# Prep data for geneformer (first data set)
adata_2_c2s = adata_2.copy()

# Prep data
adata_2_c2s = load_and_prep_data(adata = adata_2_c2s, 
                              cell_type_col = "heca_lineage", 
                              mode = "c2s",
                              organism = "Homo Sapiens")

In [12]:
# Prep data for geneformer (first data set)
adata_3_c2s = adata_3.copy()

# Prep data
adata_3_c2s = load_and_prep_data(adata = adata_3_c2s, 
                              cell_type_col = "type", 
                              mode = "c2s",
                              organism = "Homo Sapiens")

# Step 1: Create Pseudobulk

### For Geneformer

In [13]:
# For h5ad containing two different samples (duplicates)
all_pseudobulks = []
all_cell_props = []

id_list = adata_1_gf.obs['id'].unique() #since there two sample IDs here

for sample_id in id_list:
    adata_subset = adata_1_gf[adata_1_gf.obs["id"] == sample_id].copy()

    pseudobulk_temp, pseudobulk_cell_prop_temp = generate_pseudobulk(
        adata = adata_subset,
        cell_type_col = "heca_lineage",
    )

    # Optional (added sample names for easier tracking)
    pseudobulk_temp.index = [f"{sample_id}_PB_{i+1}" for i in range(0, len(pseudobulk_temp))]
    pseudobulk_cell_prop_temp.index = [f"{sample_id}_PB_{i+1}" for i in range(0, len(pseudobulk_temp))]

    all_pseudobulks.append(pseudobulk_temp)
    all_cell_props.append(pseudobulk_cell_prop_temp)

# Concatenate all results
pseudo_bulk_gf_1 = pd.concat(all_pseudobulks, axis=0, ignore_index=False)
cell_prop_1 = pd.concat(all_cell_props, axis=0, ignore_index=False)

In [14]:
#For reference containing just a sample with batch effects 
pseudo_bulk_gf_2, cell_prop_2 = generate_pseudobulk(adata = adata_2_gf,
                                       cell_type_col = "heca_lineage")

pseudo_bulk_gf_2.index = [f'{adata_2_gf.obs["id"].unique()[0]}_PB_{i+1}' for i in range(0, len(pseudo_bulk_gf_2))]
cell_prop_2.index = [f'{adata_2_gf.obs["id"].unique()[0]}_PB_{i+1}' for i in range(0, len(cell_prop_2))]

In [15]:
# Combine pseudobulk
pseudobulk_all = pd.concat([pseudo_bulk_gf_1, pseudo_bulk_gf_2], axis=0, ignore_index=False)
cell_prop_all = pd.concat([cell_prop_1, cell_prop_2], axis=0, ignore_index=False)

### For Cell2Sentence

In [16]:
# Convert using coverting coverting function
pseudobulk_all_c2s = pseudobulk_all.copy()
pseudobulk_all_c2s.columns = gene_id_name_map(gene_list = pseudobulk_all.columns,
                                      mode = "to_ensembl")

### For Cell Lines (Geneformer and Cell2Sentence)

In [53]:
#For cell lines data (default cell count per pseudobulk is 600
pb_cell_lines, cell_prop_cell_lines = generate_pseudobulk(adata = adata_3_gf,
                                       cell_type_col = "type", n_cells_per_pseudobulk = 10 )

In [54]:
pb_cell_lines_c2s = pb_cell_lines.copy()
pb_cell_lines_c2s.columns = gene_id_name_map(gene_list = pb_cell_lines.columns,
                                      mode = "to_ensembl")

In [58]:
#pb_cell_lines.to_csv("/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/cell_lines/pseudobulk/pseudobulk_for_geneformer_10_cells_per_sample.csv")
#cell_prop_cell_lines.to_csv("/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/cell_lines/pseudobulk/pseudobulk_cell_proportions_10_cells_per_sample.csv")

In [59]:
#pb_cell_lines_c2s.to_csv("/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/cell_lines/pseudobulk/pseudobulk_for_c2s_10_cells_per_sample.csv")

# Step 2: Create Signature Matrix

In [17]:
sig_mat_gf = create_signature_matrix(adata = adata_1_gf,
                                     sample_col = "id", # sample id column
                                     sample_ids = ["B4P4-CZI04N", "B4P1-CZI04N"],
                                     celltype_col = "heca_lineage",
                                     celltypes = ["Endothelial", "Mesenchymal", "Epithelial"],
                                     groupby = "heca_lineage",
                                     output_path = None)

In [18]:
sig_mat_c2s = create_signature_matrix(adata = adata_1_c2s,
                                     sample_col = "id", # sample id column
                                     sample_ids = ["B4P4-CZI04N", "B4P1-CZI04N"],
                                     celltype_col = "heca_lineage",
                                     celltypes = ["Endothelial", "Mesenchymal", "Epithelial"],
                                     groupby = "heca_lineage",
                                     output_path = None)

# Step 4: Extract Geneformer Embeddings

In [49]:
import importlib

In [65]:
from embeddings import *

In [59]:
# sig_mat_gf_embed = extract_embs(
#     bulk_df = sig_mat_gf.T,
#     mode = "geneformer",
#     temp_output_dir = "/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/", 
#     model_path= "ctheodoris/Geneformer",
#     delete_temp_files = False
# )

In [23]:
# pb_gf_embed = extract_embs(
#     bulk_df = pb_cell_lines,
#     mode = "geneformer",
#     temp_output_dir = "/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/", 
#     model_path= "ctheodoris/Geneformer",
#     delete_temp_files = True
# )

In [24]:
#pb_gf_embed.to_csv("/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/cell_lines/pseudobulk/geneformer_embeddings_10_cells_per_sample.csv")

In [25]:
#pb_gf_embed.shape

# Step 5: Extract Cell2Sentence Embeddings

In [107]:
from embeddings import *

In [117]:
sig_mat_c2s.T.head()

,MIR1302-2HG,FAM138A,OR4F5,AL627309.1,AL627309.3,AL627309.2,AL627309.5,AL627309.4,AP006222.2,AL732372.1,...,AC133551.1,AC136612.1,AC136616.1,AC136616.3,AC136616.2,AC141272.1,AC023491.2,AC007325.1,AC007325.4,AC007325.2
heca_lineage,,,,,,,,,,,,,,,,,,,,,
Endothelial,0.0,0.0,0.0,0.007855,0.000000,0.000000,0.008346,0.000491,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.002946,0.001964
Epithelial,0.0,0.0,0.0,0.009923,0.000342,0.000171,0.048246,0.003080,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000171,0.000000,0.000855,0.000171
Mesenchymal,0.0,0.0,0.0,0.003673,0.000000,0.000000,0.015215,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000525,0.000000,0.000000


In [143]:
sig_mat_c2s_embed = extract_embs(
    bulk_df = sig_mat_c2s.T,
    mode = "c2s",
    temp_output_dir = "/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/", 
    model_path= "/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/czi_sub/c2s/2b/2026-02-09-20_36_41_finetune_cell_type_prediction/checkpoint-9800/",
    delete_temp_files = True
)

WARN: more variables (32401) than observations (3)... did you mean to transpose the object (e.g. adata.T)?
WARN: more variables (32401) than observations (3), did you mean to transpose the object (e.g. adata.T)?
Saving the dataset (1/1 shards): 100%|██████████| 3/3 [00:00<00:00, 1039.31 examples/s]


Using device: cuda


ValueError: Unrecognized model in /gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/czi_sub/c2s/2b/2026-02-09-20_36_41_finetune_cell_type_prediction/checkpoint-9800/. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: albert, align, altclip, audio-spectrogram-transformer, autoformer, bark, bart, beit, bert, bert-generation, big_bird, bigbird_pegasus, biogpt, bit, blenderbot, blenderbot-small, blip, blip-2, bloom, bridgetower, bros, camembert, canine, chameleon, chinese_clip, chinese_clip_vision_model, clap, clip, clip_text_model, clip_vision_model, clipseg, clvp, code_llama, codegen, cohere, conditional_detr, convbert, convnext, convnextv2, cpmant, ctrl, cvt, dac, data2vec-audio, data2vec-text, data2vec-vision, dbrx, deberta, deberta-v2, decision_transformer, deformable_detr, deit, depth_anything, deta, detr, dinat, dinov2, distilbert, donut-swin, dpr, dpt, efficientformer, efficientnet, electra, encodec, encoder-decoder,

In [132]:
pb_c2s_embed = extract_embs(
    bulk_df = pseudobulk_all_c2s,
    mode = "c2s",
    temp_output_dir = "/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/", 
    model_path= "/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/czi_sub/c2s/2b/2026-02-09-20_36_41_finetune_cell_type_prediction/checkpoint-9700/",
    delete_temp_files = True
)

WARN: more variables (32401) than observations (3000)... did you mean to transpose the object (e.g. adata.T)?
WARN: more variables (32401) than observations (3000), did you mean to transpose the object (e.g. adata.T)?
Saving the dataset (2/2 shards): 100%|██████████| 3000/3000 [00:00<00:00, 29800.17 examples/s]


Using device: cuda
Reloading model from path on disk: /gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/c2s_model
Embedding 3000 cells using CSModel...


100%|██████████| 3000/3000 [01:05<00:00, 45.99it/s]


<b> Features to add or fix </b> <br> 
Add chunk size or batch size for both c2s and geneformer <br>
Possibly split geneformer and c2s into different functions without using a wrapper <br>
For geneformer, expose user to select which embedding dimension and type (cls vs cell) to extarct <br>
For cell2sentence, expose user to how many input dimensions to match

# Step 6: Run NNLS

### Geneformer embeddings

In [78]:
from deconvolution import *

In [18]:
pb_gf_embed.columns = "GF_" + pb_gf_embed.columns.astype(str) #Add this to the function

In [89]:
sig_mat_gf_embed.columns = "GF_" + sig_mat_gf_embed.columns.astype(str) #Add this to the function

In [26]:
cell_prop_pred = run_deconv_nnls(
    bulk_df = pb_gf_embed.T,
    signature_df = sig_mat_c2s_embed.T,
    normalize = True)

Using 1152 common features.
Running NNLS deconvolution...


### Cell2Sentence embeddings

In [135]:
pb_c2s_embed.columns = "C2S_" + pb_c2s_embed.columns.astype(str) 

In [136]:
sig_mat_c2s_embed.columns = "C2S_" + sig_mat_c2s_embed.columns.astype(str) #Add this to the function

In [137]:
cell_prop_pred_c2s = run_deconv_nnls(
    bulk_df = pb_c2s_embed.T,
    signature_df = sig_mat_c2s_embed.T,
    normalize = True)

Using 1024 common features.
Running NNLS deconvolution...


# Step 7: Evaluate 

In [126]:
from evaluation import *

### Geneformer

In [35]:
rmse_per_cell_type = compute_rmse(true_df = cell_prop_all,
                                 pred_df = cell_prop_pred,
                                 return_per_celltype = True)

rmse_per_cell_type = pd.DataFrame(rmse_per_cell_type)

rmse_per_cell_type

In [40]:
corr_per_cell_type = compute_correlation(true_df = cell_prop_all,
                                         pred_df = cell_prop_pred,
                                        return_per_celltype = True)

corr_per_cell_type = pd.DataFrame(corr_per_cell_type)

corr_per_cell_type

### Cell2Sentence

In [138]:
rmse_per_cell_type = compute_rmse(true_df = cell_prop_all,
                                 pred_df = cell_prop_pred_c2s,
                                 return_per_celltype = True)

rmse_per_cell_type = pd.DataFrame(rmse_per_cell_type)

rmse_per_cell_type

,overall,per_celltype
Endothelial,0.160382,0.146516
Mesenchymal,0.160382,0.118994
Epithelial,0.160382,0.203815


In [139]:
corr_per_cell_type = compute_correlation(true_df = cell_prop_all,
                                         pred_df = cell_prop_pred_c2s,
                                        return_per_celltype = True)

corr_per_cell_type = pd.DataFrame(corr_per_cell_type)

corr_per_cell_type

,overall,per_celltype
Endothelial,0.874035,0.920826
Mesenchymal,0.874035,0.919728
Epithelial,0.874035,0.941522


# Temp - Delete later (for testing only)

In [94]:
import sys
import importlib

# 1. Clear the 'embeddings' entry from the internal cache
if 'evaluation' in sys.modules:
    del sys.modules['evaluation']

# 2. Re-import the fresh version
import evaluation

# 3. Pull the updated functions into your current namespace
from evaluation import *

In [99]:
finetuned_models = pd.read_csv("/nfs/home/rfu/projects/deconv_language/finetunings.csv")

In [101]:
finetuned_models

,time,dir,model,param,dataset,ncell,method
0,225.570417,/gpfs/commons/groups/compbio/projects/rf_proje...,c2s,2B,czi_sub,9788,C2S_2B
1,83.050862,/gpfs/commons/groups/compbio/projects/rf_proje...,c2s,410M,czi_sub,9788,C2S_410M
2,9.281267,/gpfs/commons/groups/compbio/projects/rf_proje...,gf,316M,czi_sub,9788,GF_316M
3,107.726501,/gpfs/commons/groups/compbio/projects/rf_proje...,c2s,2B,deconvBench,4800,C2S_2B
4,70.739383,/gpfs/commons/groups/compbio/projects/rf_proje...,c2s,410M,deconvBench,4800,C2S_410M
5,15.143393,/gpfs/commons/groups/compbio/projects/rf_proje...,gf,316M,deconvBench,4800,GF_316M
6,93.318095,/gpfs/commons/groups/compbio/projects/rf_proje...,c2s,2B,GSE220608,4140,C2S_2B
7,13.215372,/gpfs/commons/groups/compbio/projects/rf_proje...,c2s,410M,GSE220608,4140,C2S_410M
8,16.258566,/gpfs/commons/groups/compbio/projects/rf_proje...,gf,316M,GSE220608,4140,GF_316M
9,1286.171553,/gpfs/commons/groups/compbio/projects/rf_proje...,c2s,2B,LIBD,56447,C2S_2B


In [141]:
finetuned_models.iloc[0, 1]

'/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/czi_sub/c2s/2b/2026-02-09-20_36_41_finetune_cell_type_prediction'

In [106]:
!ls /gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/czi_sub/c2s/2b/2026-02-09-20_36_41_finetune_cell_type_prediction

checkpoint-9700  data_split_indices_dict.pkl
checkpoint-9800  sampled_eval_indices.npy


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [146]:
print(cs.__version__)

0.0.2
